# 08 — Mechanical Wear Detection (Frequency Analysis)

Use FFT-based band power analysis on `GYR_X_DEG` to detect anomalous
high-frequency vibrations that may indicate bearing or suspension wear.

In [1]:
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go

from scooter_imu.data.loader import load_raw
from scooter_imu.data.preprocessor import preprocess
from scooter_imu.algorithms.wear_detection import WearDetector

RAW_PATH = Path("../data/raw/ScooterIMUData.csv")

In [2]:
df = preprocess(load_raw(RAW_PATH))
result = WearDetector(segment_s=30.0, freq_low=5.0, freq_high=20.0).predict(df)
print(f"Wear anomaly rows: {result['wear_anomaly'].sum()} / {len(result)}")
result[["timestamp", "gyr_x", "band_power", "wear_anomaly"]].head(10)

Wear anomaly rows: 2727 / 30000


,timestamp,gyr_x,band_power,wear_anomaly
0,2024-03-15 10:00:00.000,0.000609,1.625149e-07,False
1,2024-03-15 10:00:00.020,-0.002080,1.626734e-07,False
2,2024-03-15 10:00:00.040,0.001501,1.630469e-07,False
3,2024-03-15 10:00:00.060,0.001881,1.623146e-07,False
4,2024-03-15 10:00:00.080,-0.003902,1.639442e-07,False
5,2024-03-15 10:00:00.100,-0.002604,1.639908e-07,False
6,2024-03-15 10:00:00.120,0.000256,1.636136e-07,False
7,2024-03-15 10:00:00.140,-0.000632,1.639626e-07,False
8,2024-03-15 10:00:00.160,-0.000034,1.634688e-07,False
9,2024-03-15 10:00:00.180,-0.001706,1.632472e-07,False


In [3]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=result["timestamp"], y=result["band_power"],
                         name="Band Power", line=dict(color="gray", width=1)))
anomalies = result[result["wear_anomaly"]]
fig.add_trace(go.Scatter(x=anomalies["timestamp"], y=anomalies["band_power"],
                         mode="markers", name="Wear Anomaly",
                         marker=dict(color="red", size=6)))
fig.update_layout(title="Band Power in Wear Frequency Range (5–20 Hz)",
                  xaxis_title="Time", yaxis_title="Band Power")
fig.show()

In [4]:
# Full spectrum of a single segment
import numpy as np
from scipy import signal as sp_signal

median_dt = df["dt"].median()
fs = 1.0 / median_dt
segment = df["gyr_x"].iloc[:500].to_numpy()

freqs, psd = sp_signal.periodogram(segment, fs=fs)
fig = px.line(x=freqs, y=psd, title="Power Spectral Density (first 500 samples)",
              labels={"x": "Frequency (Hz)", "y": "PSD"})
fig.add_vrect(x0=5, x1=20, fillcolor="red", opacity=0.1, annotation_text="Wear band")
fig.show()